# Normalizing Data and Generating Insights from a Project Monitoring Database

## Project Overview

This notebook transforms a single project monitoring master table containing 80+ columns into a set of normalized relational tables following database normalization principles.

The original dataset serves as the master source of project information. Since some operational tables (e.g., contribution records, time logs, legal activities, and plan history) are not available in the original dataset, realistic mock data is generated to simulate how these tables would function in an actual project management database.

The objectives of this project are to:

- Normalize the master table into multiple relational tables.
- Generate synthetic transactional data based on realistic business rules.
- Maintain referential integrity between tables.
- Perform exploratory analysis and generate business insights from the normalized database.

> **Note:** Due to data confidentiality, mock data is used for several transactional tables while preserving the structure and relationships of the original project data.

## Read Data

In [531]:
import pandas as pd
import altair as alt
import numpy as np
import re
import random
from datetime import timedelta

random.seed(42)
np.random.seed(42)

In [532]:
term_df = pd.read_csv("data/raw/intern(term).csv", skiprows = 5)
term_df.head()

,Job \nCode,Client \nCode,Year,PD,PM,WS,Client,Subject,T,Fee,...,NAW-D.1,HZP-D.1,ABH-D.1,DTP-D.1,IFI-D.1,IMC-D.1,HZR-D.1,Service Group,Count Check,Jobcode V2
0,BCDR007,KPK01,2013,AOW,NVN,2,Komisi Pemberantasan Korupsi,BCP Development (training dan menyusun BIA),t1,"123,690,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN
1,BCDR008,IDS01,2013,AOW,MSM,2,Indosat Ooredoo,Consultant services for BCM Enhancement,t1,"199,200,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN
2,BCDR009,BCI01,2014,TSR,SHS,2,Bumiputera Sekuritas,BCP Review,t1,"8,000,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN
3,BCDR009,BCI01,2014,TSR,SHS,2,Bumiputera Sekuritas,BCP Review,t2,"12,000,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN
4,BCDR010,BNI01,2014,TSR,NVN,2,BNI Securities,BCP Review & VA,t1,"19,000,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN


## Data Cleaning

In [533]:
df = term_df.copy()

In [534]:
#Rename Columns

df = df.rename(columns={
    'Job \nCode': 'Job Code',
    'Client \nCode': 'Client Code',
    'T': 'Term',
    'WS': 'Work Status',
    'BAST #': 'BAST Status',
    'Kontrak #': 'Contract Available',
    'BAST Sent (yyyy-mm-dd)': 'BAST Sent',
    'BAST Received (yyyy-mm-dd)': 'BAST Received',
    'Invoice Sent (yyyy-mm-dd)': 'Invoice Sent',
    'Payment Estimate (yyyy-mm-dd)': 'Payment Estimate',
    'Payment Received (yyyy-mm-dd)': 'Payment Received',
    'Baseline delivery (yyyy-mm-dd)': 'Baseline delivery Date',
    'Plan delivery (yyyy-mm-dd)': 'Plan delivery Date',
    'Delayed (Days)': 'Delayed Days',
    'Last Status': 'Latest Update'

})

df.columns.tolist()

['Job Code',
 'Client Code',
 'Year',
 'PD',
 'PM',
 'Work Status',
 'Client',
 'Subject',
 'Term',
 'Fee',
 'SPK Date',
 'Deadline by Contract/Addendum',
 'Service',
 'Terms Requirements',
 'Contract Available',
 'Monitoring',
 'Delivery Status',
 'BAST Status',
 'BAST Sent',
 'BAST Received',
 'Baseline delivery Date',
 'Plan delivery Date',
 'Delayed Days',
 'Latest Update',
 'Term Status',
 'Invoice Sent',
 'Payment Estimate',
 'Payment Received',
 'Total Target Invoice',
 'Overdue',
 'Apr-26',
 'May-26',
 'Jun-26',
 'Jul-26',
 'Unscheduled',
 'On Hold',
 'ADM Status',
 'Cutoff KPI',
 'Co-PM',
 'Co-PD',
 'GTG-D',
 'THD-D',
 'TSR-D',
 'SSM-D',
 'DGS-D',
 'MKR-D',
 'RRR-D',
 'NAR-D',
 'CYH-D',
 'DDM-D',
 'AAM-D',
 'JRM-D',
 'NAW-D',
 'HZP-D',
 'ABH-D',
 'DTP-D',
 'IFI-D',
 'IMC-D',
 'HZR-D',
 'Unmapped',
 'GTG-D.1',
 'THD-D.1',
 'TSR-D.1',
 'SSM-D.1',
 'DGS-D.1',
 'MKR-D.1',
 'RRR-D.1',
 'NAR-D.1',
 'CYH-D.1',
 'DDM-D.1',
 'AAM-D.1',
 'JRM-D.1',
 'NAW-D.1',
 'HZP-D.1',
 'ABH-D.1',
 '

In [535]:
# replace missing values
df = df.replace([
    '-',
    '#N/A',
    '',
    ' '
], pd.NA)

In [536]:
# convert date columns

date_cols = [
    'SPK Date',
    'Deadline by Contract/Addendum',
    'BAST Sent',
    'BAST Received',
    'Invoice Sent',
    'Payment Estimate',
    'Payment Received'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [537]:
#inspect term column
sorted(df['Term'].dropna().unique())

['DP',
 'OPE',
 'T01',
 'T02',
 'T03',
 'T04',
 'T05',
 'T06',
 'T07',
 'T08',
 'T09',
 'T1',
 'T10',
 'T11',
 'T12',
 'T13',
 'T14',
 'T15',
 'T16',
 'T17',
 'T18',
 'T19',
 'T20',
 'T21',
 'T22',
 'T23',
 'T24',
 'T25',
 'T26',
 'T27',
 'T28',
 'T29',
 'T30',
 'T301',
 'T302',
 'T303',
 'T304',
 'T31',
 'T4',
 'T401',
 'T403',
 'T404',
 'T501',
 'T502',
 'T503',
 'T504',
 'gift',
 'printing',
 't01',
 't02',
 't03',
 't04',
 't05',
 't06',
 't07',
 't08',
 't09',
 't1',
 't10',
 't101',
 't102',
 't103',
 't11',
 't12',
 't13',
 't14',
 't15',
 't16',
 't17',
 't18',
 't19',
 't2',
 't20',
 't201',
 't202',
 't21',
 't22',
 't23',
 't24',
 't3',
 't301',
 't302',
 't4',
 't401',
 't402',
 't4a',
 't4b',
 't5',
 't501',
 't502',
 't6',
 't601',
 't602',
 't7',
 't8',
 't9']

In [538]:
# convert Term from t1, t2, ... to 1, 2, ...

def clean_term(term):
    if pd.isna(term):
        return pd.NA

    term = str(term).strip().upper()

    # Find the first sequence of digits anywhere in the string
    match = re.search(r'\d+', term)

    if not match:
        return pd.NA

    digits = match.group()

    if len(digits) <= 2:
        return int(digits)

    return int(digits[0])

df['Term'] = df['Term'].apply(clean_term).astype('Int64')

df.head()

,Job Code,Client Code,Year,PD,PM,Work Status,Client,Subject,Term,Fee,...,NAW-D.1,HZP-D.1,ABH-D.1,DTP-D.1,IFI-D.1,IMC-D.1,HZR-D.1,Service Group,Count Check,Jobcode V2
0,BCDR007,KPK01,2013,AOW,NVN,2,Komisi Pemberantasan Korupsi,BCP Development (training dan menyusun BIA),1,"123,690,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
1,BCDR008,IDS01,2013,AOW,MSM,2,Indosat Ooredoo,Consultant services for BCM Enhancement,1,"199,200,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
2,BCDR009,BCI01,2014,TSR,SHS,2,Bumiputera Sekuritas,BCP Review,1,"8,000,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
3,BCDR009,BCI01,2014,TSR,SHS,2,Bumiputera Sekuritas,BCP Review,2,"12,000,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
4,BCDR010,BNI01,2014,TSR,NVN,2,BNI Securities,BCP Review & VA,1,"19,000,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN


In [539]:
# Convert Fee and Total Target Invoice to numeric

money_cols = ['Fee', 'Total Target Invoice']

for col in money_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
    )

    df[col] = pd.to_numeric(df[col], errors='coerce')

df[['Fee', 'Total Target Invoice']].head(10)

,Fee,Total Target Invoice
0,123690000.0,NaN
1,199200000.0,NaN
2,8000000.0,NaN
3,12000000.0,NaN
4,19000000.0,NaN
5,19000000.0,NaN
6,18000000.0,NaN
7,18000000.0,NaN
8,29295000.0,NaN
9,15000000.0,NaN


In [540]:
# Convert status

status_map = {
    0: 'Starting',
    1: 'In Progress',
    2: 'Completed',
    -1: 'Cancelled'
}

status_cols = [
    'Work Status',
    'Term Status',
    'ADM Status',
    'BAST Status',

]

for col in status_cols:
    df[col] = df[col].map(status_map)


In [541]:
# Convert boolean 
df['Contract Available'] = df['Contract Available'].astype(int).astype(bool)

In [542]:
df.head()

,Job Code,Client Code,Year,PD,PM,Work Status,Client,Subject,Term,Fee,...,NAW-D.1,HZP-D.1,ABH-D.1,DTP-D.1,IFI-D.1,IMC-D.1,HZR-D.1,Service Group,Count Check,Jobcode V2
0,BCDR007,KPK01,2013,AOW,NVN,Completed,Komisi Pemberantasan Korupsi,BCP Development (training dan menyusun BIA),1,123690000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
1,BCDR008,IDS01,2013,AOW,MSM,Completed,Indosat Ooredoo,Consultant services for BCM Enhancement,1,199200000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
2,BCDR009,BCI01,2014,TSR,SHS,Completed,Bumiputera Sekuritas,BCP Review,1,8000000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
3,BCDR009,BCI01,2014,TSR,SHS,Completed,Bumiputera Sekuritas,BCP Review,2,12000000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
4,BCDR010,BNI01,2014,TSR,NVN,Completed,BNI Securities,BCP Review & VA,1,19000000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN


In [543]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5865 entries, 0 to 5864
Data columns (total 82 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Job Code                       5865 non-null   object        
 1   Client Code                    5865 non-null   object        
 2   Year                           5865 non-null   int64         
 3   PD                             5865 non-null   object        
 4   PM                             5834 non-null   object        
 5   Work Status                    5865 non-null   object        
 6   Client                         5865 non-null   object        
 7   Subject                        5863 non-null   object        
 8   Term                           5767 non-null   Int64         
 9   Fee                            5865 non-null   float64       
 10  SPK Date                       5341 non-null   datetime64[ns]
 11  Deadline by Contr

# Table 1. Contribution Partner

### Objective: Create a table that records each employee's role and percentage contribution for every project.

In [544]:
# create a list of all Job Codes
job_codes = (
    df[['Job Code']]
    .drop_duplicates()
    .reset_index(drop=True)
)

job_codes.head()

,Job Code
0,BCDR007
1,BCDR008
2,BCDR009
3,BCDR010
4,BCDR011


In [545]:
# Create the Employee Pool

employees = [
    'DTP','DLP','IFI','NFI','FYA',
    'TSR','OMW','TLW','WNA','IFS',
    'AMF','JQV','RLM','DGS','HZR'
]

In [546]:
# Generate Contribution Partner Table

contribution_partner = []

for job in job_codes['Job Code']:
    # PD
    pd_emp = random.choice(employees)

    has_copd = random.random() < 0.5

    if has_copd:
        copd_emp = random.choice([e for e in employees if e != pd_emp])

        split = random.choice([(70,30), (60,40)])

        contribution_partner.append({
            'Job Code': job,
            'Position': 'PD',
            'Employee': pd_emp,
            'Score Contribution (%)': split[0]
        })

        contribution_partner.append({
            'Job Code': job,
            'Position': 'Co-PD',
            'Employee': copd_emp,
            'Score Contribution (%)': split[1]
        })

    else:

        contribution_partner.append({
            'Job Code': job,
            'Position': 'PD',
            'Employee': pd_emp,
            'Score Contribution (%)': 100
        })


    # PM
    pm_emp = random.choice([e for e in employees if e != pd_emp])

    has_copm = random.random() < 0.5

    if has_copm:

        copm_emp = random.choice(
            [e for e in employees if e not in [pd_emp, pm_emp]]
        )

        split = random.choice([(70,30), (60,40)])

        contribution_partner.append({
            'Job Code': job,
            'Position': 'PM',
            'Employee': pm_emp,
            'Score Contribution (%)': split[0]
        })

        contribution_partner.append({
            'Job Code': job,
            'Position': 'Co-PM',
            'Employee': copm_emp,
            'Score Contribution (%)': split[1]
        })

    else:

        contribution_partner.append({
            'Job Code': job,
            'Position': 'PM',
            'Employee': pm_emp,
            'Score Contribution (%)': 100
        })

In [547]:
# Convert to DataFrame
contribution_partner = pd.DataFrame(contribution_partner)

In [548]:
# View table
contribution_partner.head(20)

,Job Code,Position,Employee,Score Contribution (%)
0,BCDR007,PD,AMF,60
1,BCDR007,Co-PD,RLM,40
2,BCDR007,PM,NFI,70
3,BCDR007,Co-PM,DGS,30
4,BCDR008,PD,AMF,100
5,BCDR008,PM,WNA,70
6,BCDR008,Co-PM,OMW,30
7,BCDR009,PD,DTP,70
8,BCDR009,Co-PD,FYA,30
9,BCDR009,PM,IFS,60


In [549]:
# Check for duplicate primary keys
contribution_partner.duplicated(
    subset=['Job Code', 'Position', 'Employee']
).sum()

0

In [550]:
# Check the contribution values
contribution_partner['Score Contribution (%)'].value_counts().sort_index()

Score Contribution (%)
30      970
40     1036
60     1036
70      970
100    2036
Name: count, dtype: int64

In [551]:
# Check that each Job sums to 100% for PD and PM
contribution_partner.groupby(
    ['Job Code', 'Position']
)['Score Contribution (%)'].sum()

Job Code        Position
BBN01-25-11-01  Co-PD        30
                Co-PM        40
                PD           70
                PM           60
BCA01-25-09-01  PD          100
                           ... 
VNI01-25-10-01  Co-PD        40
                PD           60
                PM          100
VNI01-25-10-02  PD          100
                PM          100
Name: Score Contribution (%), Length: 6048, dtype: int64

## Table 2. Time Contribution

### Objective: Record how much time each employee spent on a project.

In [552]:
# Copy Table 1. 
time_contribution = contribution_partner.copy()

In [553]:
# Rename the score column

time_contribution = time_contribution.rename(
    columns={
        'Score Contribution (%)':'Time Contribution (hours)'
    }
)

time_contribution.head()

,Job Code,Position,Employee,Time Contribution (hours)
0,BCDR007,PD,AMF,60
1,BCDR007,Co-PD,RLM,40
2,BCDR007,PM,NFI,70
3,BCDR007,Co-PM,DGS,30
4,BCDR008,PD,AMF,100


In [554]:
# Decide a constant of the total hours
TOTAL_HOURS = 40

# Replace the percentages with hours
time_contribution['Time Contribution (hours)'] = (
    time_contribution['Time Contribution (hours)']
    /100
    *TOTAL_HOURS
)

In [555]:
# Convert to integer
time_contribution['Time Contribution (hours)'] = (
    time_contribution['Time Contribution (hours)']
    .astype(int)
)

In [556]:
time_contribution.head(20)

,Job Code,Position,Employee,Time Contribution (hours)
0,BCDR007,PD,AMF,24
1,BCDR007,Co-PD,RLM,16
2,BCDR007,PM,NFI,28
3,BCDR007,Co-PM,DGS,12
4,BCDR008,PD,AMF,40
5,BCDR008,PM,WNA,28
6,BCDR008,Co-PM,OMW,12
7,BCDR009,PD,DTP,28
8,BCDR009,Co-PD,FYA,12
9,BCDR009,PM,IFS,24


In [557]:
# Check for duplicate primary keys

time_contribution.duplicated(
    subset=['Job Code','Position','Employee']
).sum()

0

In [558]:
# Validate the logic

check = time_contribution.copy()

check['Role Group'] = check['Position'].replace({
    'Co-PD': 'PD',
    'Co-PM': 'PM'
})

check.groupby(
    ['Job Code', 'Role Group']
)['Time Contribution (hours)'].sum()

Job Code        Role Group
BBN01-25-11-01  PD            40
                PM            40
BCA01-25-09-01  PD            40
                PM            40
BCA01-25-11-01  PD            40
                              ..
TSOP040         PM            40
VNI01-25-10-01  PD            40
                PM            40
VNI01-25-10-02  PD            40
                PM            40
Name: Time Contribution (hours), Length: 4042, dtype: int64

## Table 3. Legal Activity
### Objective: Records the legal document activities and their corresponding dates for each project.

In [559]:
# Create base

legal_base = (
    df[
        ['Job Code', 'SPK Date']
    ]
    .drop_duplicates('Job Code')
    .reset_index(drop=True)
)

legal_base.head()

,Job Code,SPK Date
0,BCDR007,2013-08-01
1,BCDR008,2013-10-30
2,BCDR009,2014-01-10
3,BCDR010,2014-01-20
4,BCDR011,2014-01-25


In [560]:
# Create employee pool

employees = [
    'DTP', 'DLP', 'IFI', 'NFI', 'FYA',
    'TSR', 'OMW', 'TLW', 'WNA', 'IFS',
    'AMF', 'JQV', 'RLM', 'DGS', 'HZR'
]

In [561]:
# Mock Descriptions

descriptions = [
    "Document uploaded",
    "Reviewed by Legal",
    "Waiting for approval",
    "Approved",
    "Signed by client",
    "Internal verification completed",
    "Final version uploaded"
]

In [562]:
# Generate Legal Activity table

legal_activity = []

for _, row in legal_base.iterrows():

    job = row['Job Code']
    spk_date = row['SPK Date']

    # Skip projects without an SPK Date
    if pd.isna(spk_date):
        continue

    # SPK Activities

    activities = [
        ("SPK Sent", spk_date - pd.Timedelta(days=random.randint(5, 10))),
        ("SPK Received", spk_date - pd.Timedelta(days=random.randint(1, 4))),
        ("SPK Signed", spk_date)
    ]

    # 30% chance of Addendum

    if random.random() < 0.30:

        addendum_signed = (
            spk_date +
            pd.Timedelta(days=random.randint(30, 120))
        )

        activities.extend([
            ("Addendum Sent",
             addendum_signed - pd.Timedelta(days=random.randint(5, 10))),
            ("Addendum Received",
             addendum_signed - pd.Timedelta(days=random.randint(1, 4))),
            ("Addendum Signed",
             addendum_signed)
        ])

    # Generate rows

    for activity, date in activities:

        file_name = (
            activity.lower()
            .replace(" ", "_")
            + ".pdf"
        )

        legal_activity.append({
            'Job Code': job,
            'Activity': activity,
            'Date': date,
            'PIC': random.choice(employees),
            'File Link': f'https://company.sharepoint.com/{job}/{file_name}',
            'Description': random.choice(descriptions)
        })

In [563]:
# Convert to DataFrame
legal_activity = pd.DataFrame(legal_activity)

In [564]:
# Sort by Job Code and Date
legal_activity = (
    legal_activity
    .sort_values(['Job Code', 'Date'])
    .reset_index(drop=True)
)

In [565]:
# View table
legal_activity.head(20)

,Job Code,Activity,Date,PIC,File Link,Description
0,BCA01-25-09-01,SPK Sent,2025-10-22,FYA,https://company.sharepoint.com/BCA01-25-09-01/...,Signed by client
1,BCA01-25-09-01,SPK Received,2025-10-30,WNA,https://company.sharepoint.com/BCA01-25-09-01/...,Document uploaded
2,BCA01-25-09-01,SPK Signed,2025-10-31,OMW,https://company.sharepoint.com/BCA01-25-09-01/...,Waiting for approval
3,BCA01-25-11-01,SPK Sent,2025-11-06,AMF,https://company.sharepoint.com/BCA01-25-11-01/...,Document uploaded
4,BCA01-25-11-01,SPK Received,2025-11-11,DTP,https://company.sharepoint.com/BCA01-25-11-01/...,Final version uploaded
5,BCA01-25-11-01,SPK Signed,2025-11-13,OMW,https://company.sharepoint.com/BCA01-25-11-01/...,Reviewed by Legal
6,BCDR007,SPK Sent,2013-07-24,WNA,https://company.sharepoint.com/BCDR007/spk_sen...,Reviewed by Legal
7,BCDR007,SPK Received,2013-07-29,DLP,https://company.sharepoint.com/BCDR007/spk_rec...,Signed by client
8,BCDR007,SPK Signed,2013-08-01,FYA,https://company.sharepoint.com/BCDR007/spk_sig...,Internal verification completed
9,BCDR008,SPK Sent,2013-10-21,AMF,https://company.sharepoint.com/BCDR008/spk_sen...,Approved


In [566]:
# Check for duplicate primary keys
legal_activity.duplicated(
    subset=['Job Code', 'Activity']
).sum()

0

## Table 4. PMO Activity

### Objective: Records project management activities performed throughout the project lifecycle.

In [567]:
# Base table
pmo_base = (
    legal_activity[
        legal_activity["Activity"] == "SPK Signed"
    ]
    .copy()
    .reset_index(drop=True)
)

pmo_base.head()

,Job Code,Activity,Date,PIC,File Link,Description
0,BCA01-25-09-01,SPK Signed,2025-10-31,OMW,https://company.sharepoint.com/BCA01-25-09-01/...,Waiting for approval
1,BCA01-25-11-01,SPK Signed,2025-11-13,OMW,https://company.sharepoint.com/BCA01-25-11-01/...,Reviewed by Legal
2,BCDR007,SPK Signed,2013-08-01,FYA,https://company.sharepoint.com/BCDR007/spk_sig...,Internal verification completed
3,BCDR008,SPK Signed,2013-10-30,OMW,https://company.sharepoint.com/BCDR008/spk_sig...,Waiting for approval
4,BCDR009,SPK Signed,2014-01-10,TLW,https://company.sharepoint.com/BCDR009/spk_sig...,Reviewed by Legal


In [568]:
# Mock descriptions
descriptions = [
    "BAST document created",
    "Document submitted to client",
    "Client acknowledged receipt",
    "Document archived",
    "Verified by PMO",
    "Final BAST uploaded"
]

In [569]:
# Generate PMO Activity table

pmo_activity = []

for _, row in pmo_base.iterrows():

    job = row["Job Code"]
    spk_signed_date = row["Date"]
    num_terms = random.randint(1, 5)

    for term in range(1, num_terms + 1):

        # Each term starts 60–120 days after the previous one
        term_start = (
            spk_signed_date +
            pd.Timedelta(days=(term - 1) * random.randint(60, 120))
        )

        bast_sent = (
            term_start +
            pd.Timedelta(days=random.randint(10, 20))
        )

        bast_received = (
            bast_sent +
            pd.Timedelta(days=random.randint(2, 7))
        )

        # ---------- BAST Sent ----------

        pmo_activity.append({

            "Job Code": job,
            "Term": term,
            "Activity": "BAST Sent",
            "Date": bast_sent,
            "PIC": random.choice(employees),
            "File Link": f"https://company.sharepoint.com/{job}/bast_sent.pdf",
            "Description": random.choice(descriptions)

        })

        # ---------- BAST Received ----------

        pmo_activity.append({

            "Job Code": job,
            "Term": term,
            "Activity": "BAST Received",
            "Date": bast_received,
            "PIC": pic,
            "File Link": f"https://company.sharepoint.com/{job}/bast_received.pdf",
            "Description": random.choice(descriptions)

        })

In [570]:
# Convert to DataFrame

pmo_activity = (
    pd.DataFrame(pmo_activity)
      .sort_values(["Job Code", "Term", "Date"])
      .reset_index(drop=True)
)

In [571]:
# View table
pmo_activity.head(20)

,Job Code,Term,Activity,Date,PIC,File Link,Description
0,BCA01-25-09-01,1,BAST Sent,2025-11-15,IFI,https://company.sharepoint.com/BCA01-25-09-01/...,Document submitted to client
1,BCA01-25-09-01,1,BAST Received,2025-11-18,DLP,https://company.sharepoint.com/BCA01-25-09-01/...,Document submitted to client
2,BCA01-25-09-01,2,BAST Sent,2026-02-28,WNA,https://company.sharepoint.com/BCA01-25-09-01/...,Client acknowledged receipt
3,BCA01-25-09-01,2,BAST Received,2026-03-02,DLP,https://company.sharepoint.com/BCA01-25-09-01/...,Verified by PMO
4,BCA01-25-09-01,3,BAST Sent,2026-03-29,OMW,https://company.sharepoint.com/BCA01-25-09-01/...,Final BAST uploaded
5,BCA01-25-09-01,3,BAST Received,2026-04-01,DLP,https://company.sharepoint.com/BCA01-25-09-01/...,Verified by PMO
6,BCA01-25-09-01,4,BAST Sent,2026-09-28,WNA,https://company.sharepoint.com/BCA01-25-09-01/...,Client acknowledged receipt
7,BCA01-25-09-01,4,BAST Received,2026-10-05,DLP,https://company.sharepoint.com/BCA01-25-09-01/...,Document archived
8,BCA01-25-11-01,1,BAST Sent,2025-11-30,DLP,https://company.sharepoint.com/BCA01-25-11-01/...,Client acknowledged receipt
9,BCA01-25-11-01,1,BAST Received,2025-12-02,DLP,https://company.sharepoint.com/BCA01-25-11-01/...,Document submitted to client


In [572]:
# Check for duplicate primary keys
pmo_activity.duplicated(
    subset=["Job Code", "Term", "Activity"]
).sum()

0

## Table 5. Master Baseline

### Generate the Master Baseline table by storing the fixed baseline completion date for every Job Code and Term.

In [573]:
# Create the baseline table

baseline = df[
    ["Job Code", "Term", "Baseline delivery Date", "SPK Date"]
].copy()

# Rename the column
baseline = baseline.rename(
    columns={
        "Baseline delivery Date": "Baseline Date"
    }
)

baseline.head()

,Job Code,Term,Baseline Date,SPK Date
0,BCDR007,1,NaN,2013-08-01
1,BCDR008,1,4/28/2014,2013-10-30
2,BCDR009,1,1/17/2014,2014-01-10
3,BCDR009,2,1/17/2014,2014-01-10
4,BCDR010,1,1/30/2014,2014-01-20


In [574]:
# Fix time format

baseline["Baseline Date"] = pd.to_datetime(
    baseline["Baseline Date"],
    errors="coerce",
    dayfirst=True
)

baseline.head()

/var/folders/tk/ypv35jzx7x1d9j76_spp76j80000gn/T/ipykernel_255/4288162953.py:3: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  baseline["Baseline Date"] = pd.to_datetime(


,Job Code,Term,Baseline Date,SPK Date
0,BCDR007,1,NaT,2013-08-01
1,BCDR008,1,2014-04-28,2013-10-30
2,BCDR009,1,2014-01-17,2014-01-10
3,BCDR009,2,2014-01-17,2014-01-10
4,BCDR010,1,2014-01-30,2014-01-20


In [575]:
# Find rows with missing Baseline Date
mask = baseline["Baseline Date"].isna()

# Fill missing values with SPK Date + 30–180 days
baseline.loc[mask, "Baseline Date"] = (
    baseline.loc[mask, "SPK Date"] +
    pd.to_timedelta(
        np.random.randint(30, 181, size=mask.sum()),
        unit="D"
    )
)

In [576]:
# Remove SPK Date (we only needed it to fill missing values)

baseline = baseline.drop(columns="SPK Date")

In [577]:
# Remove duplicates and sort

baseline = (
    baseline
    .drop_duplicates(subset=["Job Code", "Term"])
    .sort_values(["Job Code", "Term"])
    .reset_index(drop=True)
)

In [578]:
# View table

baseline.head(20)

,Job Code,Term,Baseline Date
0,BBN01-25-11-01,1,2026-01-09
1,BBN01-25-11-01,2,2026-01-28
2,BBN01-25-11-01,3,2026-03-10
3,BCA01-25-09-01,1,2025-11-14
4,BCA01-25-09-01,2,2025-12-03
5,BCA01-25-09-01,3,2025-12-19
6,BCA01-25-09-01,4,2025-11-24
7,BCA01-25-09-01,5,2025-12-22
8,BCA01-25-11-01,1,2026-03-26
9,BCA01-25-11-01,2,2026-04-23


In [579]:
# Check for duplicate primary keys

baseline.duplicated(
    subset=["Job Code", "Term"]
).sum()

0

## Table 6. Plan History

### Objective: Records every revision made to a project's planned delivery date for each Job Code and Term.

In [580]:
# copy the baseline table

plan_history = baseline.copy()

In [581]:
# Generate revisions

random.seed(42)

history_rows = []

for _, row in plan_history.iterrows():

    job = row["Job Code"]
    term = row["Term"]
    baseline_date = pd.to_datetime(row["Baseline Date"])

    # Randomly decide how many revisions
    num_revisions = random.randint(1, 4)

    current_date = baseline_date

    for revision in range(1, num_revisions + 1):

        # Shift plan date by -7 to +14 days
        current_date = current_date + pd.Timedelta(
            days=random.randint(-7, 14)
        )

        history_rows.append({
            "Job Code": job,
            "Term": term,
            "Revision": revision,
            "Plan Date": current_date
        })

In [582]:
# Create the dataframe

plan_history = pd.DataFrame(history_rows)

In [583]:
# Sort

plan_history = (
    plan_history
    .sort_values(["Job Code", "Term", "Revision"])
    .reset_index(drop=True)
)

In [584]:
plan_history.head(20)

,Job Code,Term,Revision,Plan Date
0,BBN01-25-11-01,1,1,2026-01-02
1,BBN01-25-11-01,2,1,2026-01-28
2,BBN01-25-11-01,2,2,2026-01-28
3,BBN01-25-11-01,2,3,2026-01-25
4,BBN01-25-11-01,3,1,2026-03-24
5,BCA01-25-09-01,1,1,2025-11-25
6,BCA01-25-09-01,2,1,2025-11-27
7,BCA01-25-09-01,2,2,2025-11-20
8,BCA01-25-09-01,2,3,2025-11-15
9,BCA01-25-09-01,2,4,2025-11-14


In [585]:
# Check the number of revisions per job

plan_history.groupby(
    ["Job Code", "Term"]
)["Revision"].max().value_counts()

Revision
4    1476
2    1422
1    1406
3    1375
Name: count, dtype: int64

In [586]:
# Check for duplicate primary keys

plan_history.duplicated(
    subset=["Job Code", "Term", "Revision"]
).sum()

0

## Table 7. Finance
### Objective: a Finance Activity table that records the financial progress of each Job Code and Term, including invoice issuance, billing, and payment.

In [587]:
# Create baseline table

finance_base = baseline.copy()

In [588]:
# Finance activity options

finance_pic = [
    "TSR", "MRK", "DGS", "OMW", "AOW",
    "NVN", "SHS", "SSM", "AMF", "RLM"
]

finance_desc = [
    "Invoice issued to client",
    "Invoice verified",
    "Invoice successfully paid",
    "Payment confirmation received",
    "Finance approval completed"
]

In [589]:
# Generate the finance table

finance_activity = []

for _, row in finance_base.iterrows():

    job = row["Job Code"]
    term = row["Term"]

    baseline_date = pd.to_datetime(row["Baseline Date"])

    # Invoice Sent
    invoice_sent = baseline_date + pd.Timedelta(
        days=random.randint(1, 7)
    )

    finance_activity.append({
        "Job Code": job,
        "Term": term,
        "Activity": "Invoice Sent",
        "Date": invoice_sent,
        "PIC": random.choice(finance_pic),
        "Description": random.choice(finance_desc)
    })

    # 90% chance invoice is billed
    if random.random() < 0.90:

        invoice_billed = invoice_sent + pd.Timedelta(
            days=random.randint(3, 14)
        )

        finance_activity.append({
            "Job Code": job,
            "Term": term,
            "Activity": "Invoice Billed",
            "Date": invoice_billed,
            "PIC": random.choice(finance_pic),
            "Description": random.choice(finance_desc)
        })

        # 85% chance invoice is paid
        if random.random() < 0.85:

            invoice_paid = invoice_billed + pd.Timedelta(
                days=random.randint(7, 45)
            )

            finance_activity.append({
                "Job Code": job,
                "Term": term,
                "Activity": "Invoice Paid",
                "Date": invoice_paid,
                "PIC": random.choice(finance_pic),
                "Description": random.choice(finance_desc)
            })

In [590]:
# Convert to DataFrame

finance_activity = pd.DataFrame(finance_activity)

In [591]:
# Sort

finance_activity = (
    finance_activity
    .sort_values(["Job Code", "Term", "Date"])
    .reset_index(drop=True)
)

In [592]:
finance_activity.head(20)

,Job Code,Term,Activity,Date,PIC,Description
0,BBN01-25-11-01,1,Invoice Sent,2026-01-14,SSM,Finance approval completed
1,BBN01-25-11-01,1,Invoice Billed,2026-01-17,AMF,Invoice verified
2,BBN01-25-11-01,1,Invoice Paid,2026-01-25,AOW,Finance approval completed
3,BBN01-25-11-01,2,Invoice Sent,2026-01-29,SHS,Payment confirmation received
4,BBN01-25-11-01,2,Invoice Billed,2026-02-11,MRK,Invoice verified
5,BBN01-25-11-01,2,Invoice Paid,2026-03-14,RLM,Invoice issued to client
6,BBN01-25-11-01,3,Invoice Sent,2026-03-13,RLM,Payment confirmation received
7,BBN01-25-11-01,3,Invoice Billed,2026-03-26,MRK,Invoice verified
8,BBN01-25-11-01,3,Invoice Paid,2026-05-03,RLM,Invoice successfully paid
9,BCA01-25-09-01,1,Invoice Sent,2025-11-21,DGS,Finance approval completed


In [593]:
# Check for duplicate primary keys

finance_activity.duplicated(
    subset=["Job Code", "Term", "Activity"]
).sum()

0

### Save all tables as csv

In [ ]:
# Dictionary of all normalized tables
tables = {
    "contribution_partner": contribution_partner,
    "time_contribution": time_contribution,
    "legal_activity": legal_activity,
    "pmo_activity": pmo_activity,
    "master_baseline": baseline,
    "plan_history": plan_history,
    "finance_activity": finance_activity
}

# Export each table as a CSV
for name, table in tables.items():
    table.to_csv(f"data/processed/{name}.csv", index=False)

print("✅ All normalized tables have been saved to data/processed!")

✅ All normalized tables have been saved to data/processed!


### Save all tables as excel

In [598]:
from openpyxl.styles import NamedStyle

os.makedirs("data/processed", exist_ok=True)

tables = {
    "contribution_partner": contribution_partner,
    "time_contribution": time_contribution,
    "legal_activity": legal_activity,
    "pmo_activity": pmo_activity,
    "master_baseline": baseline,
    "plan_history": plan_history,
    "finance_activity": finance_activity
}

output_file = "data/processed/normalized_tables.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet_name, table in tables.items():

        df = table.copy()

        # Convert every datetime column to date (still a date object, not string)
        for col in df.select_dtypes(include=["datetime64[ns]"]).columns:
            df[col] = df[col].dt.date

        df.to_excel(writer, sheet_name=sheet_name, index=False)

        ws = writer.sheets[sheet_name]

        # Apply Excel date format
        for col in df.select_dtypes(include=["object"]).columns:
            if len(df) > 0 and hasattr(df[col].iloc[0], "year"):
                col_idx = df.columns.get_loc(col) + 1
                for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx,
                                         min_row=2, max_row=ws.max_row):
                    for c in cell:
                        c.number_format = "yyyy-mm-dd"